In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import json
from pathlib import Path
from ast import literal_eval

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from utils.tools import aggregate_results

In [3]:
def get_suite(row):

    n_demos = row["number of demonstrations"]
    type_demos = row["type of demonstrations"][0:3]
    instr = "impl" if row["use instructions"] == "no" else "expl"

    return f"{n_demos}-{type_demos}-{instr}"

def capitalize(s):
    return s[0].upper() + s[1:]

def order_suites(strings):
    return sorted(strings, key=lambda s: (
        s.split('-')[2],  # implicit-excplict
        s.split('-')[0],  # num demos
        s.split('-')[1],  # demo type
    ))

def order_models(strings):
    return sorted(strings, key=lambda s: (
        int(s.split('-')[1][:-1]),  # Param count, B dropped
        #s.split('-')[0],  # Model name
    ), reverse=True)

def highlight_extrema(s, use_italics=False, exclude_index=None):

    comp_list = s
    if exclude_index is not None:
        comp_list = s.drop(labels=exclude_index)
    
    styles = []
    for v in s:      
        style = ''
        if not isinstance(v, (int, float, np.floating)):
            pass
        elif v not in comp_list.values:
            pass
        elif v == comp_list.max():
            style = 'font-weight: bold;'
        elif v == comp_list.min():
            style = 'font-weight: bold; font-style: italic;'
        styles.append(style)
    
    return styles

def percentage_improvement(df, against="0-non-expl"):
    if against in ["", "max"] or against is None:
        return df.apply(lambda col: (col - col.max()) / col.max(), axis=0)

    if against == "min":
        return df.apply(lambda col: (col.max() - col.min()) / col.min(), axis=0) 
        
    return df.apply(lambda col: (col.loc[col.index != against].max() - col.loc[against]) / col.loc[against], axis=0)
    

def diff_to_baseline(df, t="max", perc=False):

    against = "0-non-expl"
    
    if t == "max":
        fun = pd.Series.max
    else:
        fun = pd.Series.min

    return df.apply(lambda col: (fun(col.loc[col.index != against]) - col.loc[against]) / (1 if not perc else col.loc[against]), axis=0)



def value_improvement(df, against="0-non-expl"):
    if against in ["", "max"] or against is None:
        return df.apply(lambda col: (col - col.max()), axis=0)

    if against == "min":
        return df.apply(lambda col: col.max() - col.min(), axis=0) 
        
    
    return df.apply(lambda col: col.loc[col.index != against].max() - col.loc[against], axis=0)   

def sig3(x):
    if isinstance(x, (int, float, np.floating)):
        return f"{x:.3g}"
    return x

def sig2(x):
    if isinstance(x, (int, float, np.floating)):
        return f"{x:.2g}"
    return x

In [4]:
### Load F1 data

metric_cols_f1 = ["theme f1 mean", "topic f1 mean", "concept f1 mean"]

res = pd.read_csv("./data/metrics.csv", sep=";")
res = res.sort_values(by=["use instructions", "number of demonstrations", "type of demonstrations"])
res["suite"] = res.apply(get_suite, axis=1)

In [5]:
### Create top 10 suites for each label 
# IN USE!

metric_cols = ["Theme", "Topic", "Concept", "Sum"]

f1s = res[["model", "suite"] + metric_cols_f1]
f1s = f1s.rename(columns={col: col.split(" ")[0] for col in metric_cols_f1})

f1s["sum"] = f1s["theme"] + f1s["topic"] + f1s["concept"]
f1s = f1s.rename(columns={col: capitalize(col) for col in f1s.columns})


top10s = [
    f1s
        .sort_values(by=col, ascending=False)[:10]
    for col in metric_cols
]

for df in top10s:
    pass
    #print(df.to_latex(index=False, float_format="%.3g", column_format="ll|rrrr"))

In [15]:
### Create one table per label, where best of each model is presented
# In Use!

tops_per_model = [
    f1s
        .loc[f1s.groupby('Model').idxmax()[col]]
        .sort_values(by="Model", ascending=True)
    for col in metric_cols
]

for df in tops_per_model:
    pass
    #print(df.to_latex(index=False, float_format="%.3g", column_format="lr|lr|lr|lr"))

In [16]:
### Table with models ordered by their best score per label
# In Use!

models_ordered_by_scores = {}

for i, col in enumerate(metric_cols):
    best_scores_for_col = tops_per_model[i]
    ordered = best_scores_for_col.sort_values(by=col, ascending=False)
    models_ordered_by_scores[col] = ordered["Model"].to_list()
    diffs = value_improvement(ordered.loc[:, [col]], None)[col].to_list()
    diffs[0] = ordered[col].max()   
    models_ordered_by_scores[f"{col} F1 (Top, $\\Delta$)"] = diffs

print(pd.DataFrame(models_ordered_by_scores).to_latex(index=False, float_format="%.3g", column_format="lr|lr|lr|lr"))

\begin{tabular}{lr|lr|lr|lr}
\toprule
Theme & Theme F1 (Top, $\Delta$) & Topic & Topic F1 (Top, $\Delta$) & Concept & Concept F1 (Top, $\Delta$) & Sum & Sum F1 (Top, $\Delta$) \\
\midrule
Llama-70B & 0.949 & Llama-70B & 0.978 & Llama-8B & 0.902 & Llama-70B & 2.8 \\
Qwen3-32B & -0.00243 & Qwen3-32B & -0.00721 & Llama-70B & -0.0275 & Llama-8B & -0.127 \\
Mistral-24B & -0.0456 & Llama-8B & -0.0598 & Qwen3-32B & -0.159 & Qwen3-32B & -0.225 \\
Llama-8B & -0.0531 & Mistral-24B & -0.0695 & Mistral-24B & -0.178 & Mistral-24B & -0.284 \\
Qwen3-8B & -0.0974 & Qwen3-8B & -0.0811 & Mistral-7B & -0.21 & Qwen3-8B & -0.575 \\
Mistral-7B & -0.179 & Mistral-7B & -0.179 & Qwen3-8B & -0.226 & Mistral-7B & -0.577 \\
\bottomrule
\end{tabular}



In [8]:
### Table with models ordered by their baseline score per label
## IN USE!

baselines = f1s.groupby(by=["Model", "Suite"]).max().loc[(slice(None), "0-non-expl"), :]
baselines.index = baselines.index.get_level_values(0)
baselines = baselines.reset_index()

model_baselines_ordered = {}

for i, col in enumerate(metric_cols):
    ordered = baselines.sort_values(by=col, ascending=False)
    model_baselines_ordered[col] = ordered["Model"].to_list()

    diffs = value_improvement(ordered.loc[:, [col]], None)[col].to_list()
    diffs[0] = ordered[col].max() 
    
    model_baselines_ordered[f"{col} F1 (Top, $\Delta$)"] = diffs
    
#print(pd.DataFrame(model_baselines_ordered).to_latex(index=False, float_format="%.3g", column_format="lr|lr|lr|lr"))

In [10]:
### Baseline, max, min for each model and label
# In Use!
models = f1s["Model"].unique()

baselines = f1s["Suite"] == "0-non-expl"

max_values = f1s[["Model"] + metric_cols].groupby(by="Model").idxmax()
min_values = f1s[["Model"] + metric_cols].groupby(by="Model").idxmin()

best_per_label = pd.concat(
    [
        f1s[baselines],
        f1s.loc[max_values.values.reshape(-1)],
        f1s.loc[min_values.values.reshape(-1)],
    ]
).groupby(["Model", "Suite"]).max()

for model in models:
    pass

    per_model = best_per_label.loc[model]
    
    #print("%" + model)
    #print(
    #    per_model.style.apply(
    #        lambda col: highlight_extrema(
    #            col,
    #            True,
    #            exclude_index=[idx for idx in per_model.index if "Diff" in idx]
    #        )
    #    ).format(sig3).to_latex(convert_css=True)
    #)

In [11]:
### Differences: max to baseline, min to baseline, max to min
# Not in use

for model in models:
    pass

    per_model = best_per_label.loc[model]

    best = diff_to_baseline(per_model, "max", False)
    best_p = diff_to_baseline(per_model, "max", True)

    worst = diff_to_baseline(per_model, "min", False)
    worst_p = diff_to_baseline(per_model, "min", True)

    rel = value_improvement(per_model, "min")
    rel_p = percentage_improvement(per_model, "min")

    diffs = pd.DataFrame(
        {
            "Diff hi/bl": best,
            "Diff hi/bl (\\%)": best_p,
            "Diff lo/bl": worst,
            "Diff lo/bl (\\%)": worst_p,
            "Diff hi/lo": rel,
            "Diff hi/lo (\\%)": rel_p,
        }
    )
    
    #print("%" + model)
    #print(
    #    diffs.T.style.format(sig2).to_latex()
    #)

In [245]:
path = "./data/mcnemar.json"
mcnemar = pd.read_json(path)
mcnemar.columns = [" ".join(literal_eval(tup)).strip() for tup in mcnemar.columns]
mcnemar = mcnemar[["Model", "Suite"] + [col for col in mcnemar.columns if any(s in col for s in ["Holm", "$S$"])]]

f1s_with_p_values = f1s.merge(mcnemar, left_on=["Model", "Suite"], right_on=["Model", "Suite"])
f1s_with_p_values = f1s_with_p_values.set_index(["Model", "Suite"])
f1s_with_p_values = f1s_with_p_values.sort_index(axis=0)
f1s_with_p_values = f1s_with_p_values.rename(columns={
    col: f"{col} Top F1"
    for col in metric_cols
})

In [246]:
baseline_values = f1s_with_p_values.loc[(slice(None), "0-non-expl"), f1s_with_p_values.columns[:4]]
baseline_values.index = [idx[0] for idx in baseline_values.index]

In [247]:
baseline_values

,Theme Top F1,Topic Top F1,Concept Top F1,Sum Top F1
Llama-70B,0.944742,0.977860,0.857651,2.780253
Llama-8B,0.815870,0.776810,0.749048,2.341729
Mistral-24B,0.886340,0.892556,0.662788,2.441684
Mistral-7B,0.746351,0.785973,0.053761,1.586085
Qwen3-32B,0.935841,0.966240,0.569350,2.471431
Qwen3-8B,0.851361,0.896785,0.410304,2.158449


In [248]:
tops_per_model = [
    f1s_with_p_values
        .loc[f1s_with_p_values.groupby('Model').idxmax()[f"{col} Top F1"]]
        .sort_values(by="Model", ascending=True)
    for col in metric_cols
]

In [249]:
to_merge = []

for i, top_score_df in enumerate(tops_per_model):
    col = metric_cols[i]
    col_f1 = f"{metric_cols[i]} Top F1"

    top_score_df[f"{col } Baseline"] = baseline_values[col_f1].values
    
    cur_df = top_score_df.reset_index()
    cur_df.loc[:, "Sum $S$"] = cur_df[
        [
            f"{c} $S$"
            for c in metric_cols[:-1]
        ]
    ].sum(axis=1)
    cur_df.loc[:, f"{col} $\\Delta$"] = cur_df[col_f1] - cur_df[f"{col} Baseline"]
    cur_df = cur_df[["Model", "Suite", f"{col} Baseline", col_f1,  f"{col} $\\Delta$", f"{col} $S$"]]

    to_merge.append(cur_df)

final = pd.concat(to_merge, axis=1).set_index("Model")
final.index = [idx[0] for idx in final.index]

In [250]:
s_cols = [col for col in final.columns if "$S$" in col]

final[s_cols] = final[s_cols].map(lambda e: 
                  ""
                  if np.isnan(e)
                  else "ns" if e < 1 
                  else int(e) * "*")

final.columns = pd.MultiIndex.from_arrays(
    [
        [col for label in metric_cols for col in [label] * 5],
        [col.replace("Theme", "").strip() for col in final.columns[:5]] * 4
    ],
    names=["Label", "Metric"]
)

In [256]:
latex = final.loc[:, final.columns[:10]].to_latex(
    column_format= "l|rllll|rllll",
    index=True,
    float_format="%.3g"
) + "\n" + final.loc[:, final.columns[10:]].to_latex(
    column_format= "l|rllll|rllll",
    index=True,
    float_format="%.3g"
)

print(latex)

\begin{tabular}{l|rllll|rllll}
\toprule
Label & \multicolumn{5}{r}{Theme} & \multicolumn{5}{r}{Topic} \\
Metric & Suite & Baseline & Top F1 & $\Delta$ & $S$ & Suite & Baseline & Top F1 & $\Delta$ & $S$ \\
\midrule
Llama-70B & 6-pos-expl & 0.945 & 0.949 & 0.00402 & ns & 0-non-expl & 0.978 & 0.978 & 0 &  \\
Llama-8B & 6-mix-expl & 0.816 & 0.896 & 0.0798 & * & 6-pos-expl & 0.777 & 0.918 & 0.141 & * \\
Mistral-24B & 6-mix-expl & 0.886 & 0.903 & 0.0168 & * & 1-pos-expl & 0.893 & 0.908 & 0.0158 & ns \\
Mistral-7B & 6-mix-expl & 0.746 & 0.77 & 0.0232 & * & 6-mix-expl & 0.786 & 0.799 & 0.0132 & * \\
Qwen3-32B & 6-mix-expl & 0.936 & 0.946 & 0.0105 & * & 6-mix-expl & 0.966 & 0.971 & 0.00441 & ns \\
Qwen3-8B & 0-non-expl & 0.851 & 0.851 & 0 &  & 0-non-expl & 0.897 & 0.897 & 0 &  \\
\bottomrule
\end{tabular}

\begin{tabular}{l|rllll|rllll}
\toprule
Label & \multicolumn{5}{r}{Concept} & \multicolumn{5}{r}{Sum} \\
Metric & Suite & Baseline & Top F1 & $\Delta$ & $S$ & Suite & Baseline & Top F1 & $\De